# Bots

> Command routing over REST + Gateway, with voice conveniences

In [ ]:
#| default_exp bot

In [ ]:
from cordslite.core import *
from cordslite.gateway import *
from cordslite.voice import *
from fastcore.utils import *

import asyncio

In [ ]:
from IPython.display import Audio

import wave

In [ ]:
dc = DiscordClient()
gld = await dc.guild('1327046393453613076')
chs = await gld.channels()
vch = chs[3]
intents = (1 << 0) | (1 << 7) | (1 << 9) | (1 << 15) # GUILDS | VOICE_EVENTS | GUILD_MESSAGES | MESSAGE_CONTENT

`Bot` ties together `DiscordClient` (REST) and `GatewayClient` (events) into a single object with a decorator-based command router. Commands are registered with `@bot.cmd` — the function name becomes the command name, prefixed with `!` in Discord. So `def echo` responds to `!echo`. Every command handler takes two arguments: the `Message` object and a string of everything the user typed after the command name (empty string if nothing).

The `_on_msg` handler ignores messages from the bot itself to prevent infinite loops — a common gotcha with Discord bots. It splits the message into command name + args, so `!echo hello world` passes `"hello world"` as the `args` string to the handler.

In [ ]:
class Bot:
    "Discord bot with command routing"
    def __init__(self, intents, **kw):
        self.dc = DiscordClient(**kw)
        self.gc = GatewayClient(intents, self.dc)
        self.cmds = {}
        self.errors = []
        self._on_err = None

    def cmd(self, f):
        "Register a command handler — function name becomes !name"
        self.cmds[f.__name__] = f
        return f

    async def _on_msg(self, msg):
        if msg.author['id'] == self.gc.user_id: return
        parts = msg.content.split(maxsplit=1)
        if not parts or not parts[0].startswith('!'): return
        name = parts[0][1:]
        if name not in self.cmds: return
        try:
            res = self.cmds[name](msg, parts[1] if len(parts) > 1 else '')
            if asyncio.iscoroutine(res): await res
        except Exception as e:
            self.errors.append(e)
            if self._on_err: await self._on_err(msg, e)

    async def start(self):
        "Connect to gateway and start listening"
        await self.gc.start()
        self.gc.on('MESSAGE_CREATE', self._on_msg)

    async def stop(self): await self.gc.stop()
    def __repr__(self): return f"Bot(cmds={list(self.cmds.keys())})"

In [ ]:
bot = Bot(intents)
await bot.start()

Commands can be registered at any time — even after `bot.start()`. This works because `@bot.cmd` just adds the function to a dict; the message handler looks up commands dynamically on each message.

In [ ]:
@bot.cmd
async def echo(msg, args): await (await msg.channel()).send(f'You said: {args}')

In [ ]:
bot

Bot(cmds=['echo'])

Errors in command handlers are caught and stored in `bot.errors` — useful for debugging in dynamic environments like solveit where you can inspect the list after the fact. For real-time handling (e.g. notifying the user in Discord), register a handler with `@bot.on_error`. Both mechanisms work simultaneously.

In [ ]:
@patch
def on_error(self:Bot, f):
    self._on_err = f
    return f

In [ ]:
@bot.on_error
async def handle_err(msg, e): print('error')

In [ ]:
@bot.cmd
def err(msg): raise Exception('test')

In [ ]:
bot.errors

[]

Voice integration reuses the existing `VoiceClient` — `Bot` just provides convenience methods to manage the lifecycle. The bot can only be in one voice channel at a time.

In [ ]:
@patch
async def join_voice(self:Bot, channel):
    "Join a voice channel and return VoiceClient"
    self.vc = VoiceClient(self.gc, channel.guild_id, channel)
    await self.vc.join()
    return self.vc

@patch
async def leave_voice(self:Bot):
    "Leave the current voice channel"
    if self.vc: await self.vc.leave()
    self.vc = None

In [ ]:
vc = await bot.join_voice(vch); vc

VoiceClient(self.ch=Channel(id=1327046393453613080, name='General', type=2))

voice json 11 {'user_ids': ['346450717025894400']}


voice json 18 {'user_id': '346450717025894400', 'flags': 2}


voice json 20 {'user_id': '346450717025894400', 'platform': 0}


voice json 15 {'any': 100}


In [ ]:
vc.start_recording()

'/tmp/recording.mp3'

In [ ]:
pth = vc.stop_recording()
await bot.leave_voice()

In [ ]:
# Audio(pth)

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()